# 17 — 大图 GNN：邻居采样、mini-batch 与 GraphSAGE

前 16 课大多采用 **full-batch**：一次前向传播读取整张图。这样最容易理解公式，却无法自然扩展到百万节点和千万条边。本课不引入更花哨的模型，而是解决一个真正的 GNN 工程问题：**节点互相连接时，mini-batch 到底应该怎样取？**

本课使用完整公开数据 `ogbn-arxiv`，比较不使用图的 MLP 与使用 `NeighborLoader` 训练的 GraphSAGE。`RUN_FULL=False` 只减少 epoch、seed 和 fanout 组合，不截断数据集。

## 1. 学习目标

完成本课后，你应该能解释：

1. 为什么普通的节点 mini-batch 会切断消息传递；
2. `num_neighbors=[15, 10, 5]` 中每个数字的含义；
3. sampled nodes 与 seed/target nodes 有什么区别；
4. 为什么只对 batch 前 `batch_size` 个节点计算监督损失；
5. 为什么训练可以随机采样，最终验证却应使用确定性的全邻居推理；
6. fanout 怎样影响速度、显存、采样方差和信息覆盖。

## 2. 为什么图不能像图片一样随便切 batch？

普通监督学习的样本互相独立；图中的节点不是。如果只把目标节点集合 $B$ 放入 batch，一层 GNN 还需要 $B$ 的邻居，两层还需要邻居的邻居。若平均度为 $d$，最坏情况下 $L$ 层计算图接近 $|B|d^L$，这叫 **neighbor explosion（邻居爆炸）**。

邻居采样把第 $l$ 层最多读取的邻居数限制为 $s_l$：

$$|V_{batch}| \lesssim |B|\left(1+s_1+s_1s_2+\cdots+\prod_{l=1}^{L}s_l\right).$$

注意这是上界直觉，不是精确节点数：不同种子会共享邻居，重复节点会被合并。fanout 越大越接近全邻居聚合，但显存和计算也越大。

## 3. GraphSAGE 的数学原理

均值版 GraphSAGE 在第 $l$ 层先聚合采样到的邻居：

$$m_v^{(l)} = \operatorname{mean}_{u\in\mathcal{S}_l(v)} h_u^{(l-1)},$$

再分别变换自身与邻居信息：

$$h_v^{(l)} = \sigma\left(W_{self}^{(l)}h_v^{(l-1)} + W_{neigh}^{(l)}m_v^{(l)} + b^{(l)}\right).$$

$\mathcal{S}_l(v)$ 是本轮随机采到的邻居，而不是永久固定的邻居。多轮训练相当于让节点看到不同的局部视野。GraphSAGE 学的是共享聚合函数，不是每个节点的 ID embedding，因此能对具有特征的新节点做 **inductive（归纳式）** 推理。

## 4. 数据与时间划分

OGBN-Arxiv 是论文引用图：节点是 arXiv 论文，边是引用，128 维特征来自标题和摘要，标签是 40 个研究主题。官方划分按论文发表时间构造：较早论文训练，之后论文验证，更晚论文测试。它比随机切分更贴近现实——用过去论文预测未来论文类别。

原始引用是有方向的。本课与常见 OGB 基线一致，把边转成双向消息边：这不是说引用关系在现实中对称，而是允许被引用论文与引用论文交换表示。数据加载器保留官方 train/valid/test 索引，不自行随机切分。

In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from crosscity.data.scalable_node_classification import load_ogbn_arxiv
from crosscity.models.scalable_gnn import SampledGraphSAGE, ScalableMLP
from crosscity.training.scalable_node_classification import (
    make_neighbor_loaders,
    train_sampled_graphsage,
    train_scalable_mlp,
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('torch:', torch.__version__, 'cuda runtime:', torch.version.cuda)

## 5. PyG 采样加速后端

把模型和 tensor 放到 CUDA 只会加速神经网络计算；`NeighborLoader` 的 CPU/图采样还需要 `pyg-lib` 或 `torch-sparse`。它们必须与服务器的 PyTorch 和 CUDA ABI 匹配，因此项目不能硬编码某个 CUDA wheel。下面生成与你当前环境对应的安装命令，请复制到终端执行后重启 kernel。CPU 环境同样可以从 PyG wheel 仓库安装对应版本。

In [ ]:
torch_version = torch.__version__.split('+')[0]
cuda_tag = f"cu{torch.version.cuda.replace('.', '')}" if torch.version.cuda else 'cpu'
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
print(f'python -m pip install pyg_lib -f {wheel_url}')

backends = {}
for package in ('pyg_lib', 'torch_sparse'):
    try:
        module = __import__(package)
        backends[package] = getattr(module, '__version__', 'installed')
    except ImportError:
        backends[package] = None
backends

## 6. 下载受限服务器的文件放置

正常情况下 OGB 会自动下载。如果远程服务器无法联网，可以在本地下载 `ogbn_arxiv.zip`，把压缩包上传到项目根目录，然后在服务器终端执行：

```bash
mkdir -p data/raw/ogb
unzip ogbn_arxiv.zip -d data/raw/ogb
ls data/raw/ogb/ogbn_arxiv/raw
```

不要把解压后的公开数据提交到 Git。

In [ ]:
RUN_FULL = False
ROOT = Path('../data/raw/ogb')
SEEDS = [42, 43, 44] if RUN_FULL else [42]
MAX_EPOCHS = 30 if RUN_FULL else 5
PATIENCE = 8 if RUN_FULL else 5
BATCH_SIZE = 1024
INFERENCE_BATCH_SIZE = 4096
NUM_WORKERS = 4 if RUN_FULL else 0
FANOUT_CONFIGS = {
    'small-[5,5,5]': [5, 5, 5],
    'wide-[15,10,5]': [15, 10, 5],
}
if not RUN_FULL:
    FANOUT_CONFIGS = {'small-[5,5,5]': [5, 5, 5]}

In [ ]:
graph = load_ogbn_arxiv(ROOT)
num_classes = int(graph.y.max()) + 1
audit = {
    'nodes': graph.num_nodes,
    'directed_message_edges_after_ToUndirected': graph.num_edges,
    'features': graph.num_features,
    'classes': num_classes,
    'train_nodes': int(graph.train_mask.sum()),
    'valid_nodes': int(graph.valid_mask.sum()),
    'test_nodes': int(graph.test_mask.sum()),
}
pd.Series(audit, name='ogbn-arxiv')

## 7. 观察一个 sampled mini-batch

`NeighborLoader` 返回的 `batch` 不是一组孤立目标节点，而是一张重新编号的局部计算图：

- `batch.batch_size`：真正需要监督的 seed 节点数；
- `batch.n_id`：局部节点映射回全图节点 ID；
- `batch.x`：seed 加多层邻居的特征；
- `batch.edge_index`：局部重编号后的采样边；
- 前 `batch.batch_size` 个节点一定是 seed，所以 loss 只读取 logits 的这一段。

其余节点是计算 seed 表示所需的上下文，不应被误当作本 batch 的训练目标。

In [ ]:
train_loader, inference_loader = make_neighbor_loaders(
    graph,
    num_neighbors=[5, 5, 5],
    batch_size=BATCH_SIZE,
    inference_batch_size=INFERENCE_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
sample = next(iter(train_loader))
pd.Series({
    'seed_nodes': sample.batch_size,
    'all_sampled_nodes': sample.num_nodes,
    'sampled_edges': sample.num_edges,
    'x_shape': tuple(sample.x.shape),
    'first_seed_global_id': int(sample.n_id[0]),
})

## 8. 为什么训练采样、评价全邻居？

训练阶段随机采样既控制成本，也像一种结构 dropout。但若 validation 每次也随机采样，同一个 checkpoint 会因抽到不同邻居而得到不同分数。

本课使用 **layer-wise inference**：一次只把一层所需的局部边放到设备，对所有节点计算该层表示并存回 CPU，然后进入下一层。每层使用 `num_neighbors=[-1]`，即读取完整一跳邻居。这样不必把整张图的全部中间激活同时放进 GPU，同时评价是确定且接近部署时的全邻居结果。

## 9. 实验：MLP 与 sampled GraphSAGE

MLP 是不可省略的无图基线。如果 GraphSAGE 没有超过 MLP，可能是引用邻居没有增量信息，也可能是 fanout 太小、层数不合适或采样噪声太大。正式模式会比较两个 fanout；不要根据 test 选择 fanout。

In [ ]:
records = []
histories = {}

for seed in SEEDS:
    seed_everything(seed)
    mlp_result = train_scalable_mlp(
        ScalableMLP(graph.num_features, 256, num_classes),
        graph, device=DEVICE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
    )
    records.append({
        'seed': seed, 'model': 'MLP', 'fanout': 'none',
        'best_epoch': mlp_result.best_epoch,
        'validation_accuracy': mlp_result.validation_accuracy,
        'test_accuracy': mlp_result.test_accuracy,
        'seconds': mlp_result.seconds, 'peak_cuda_mb': mlp_result.peak_cuda_mb,
    })
    histories[(seed, 'MLP')] = mlp_result.history

    for fanout_name, fanouts in FANOUT_CONFIGS.items():
        seed_everything(seed)
        sage_result = train_sampled_graphsage(
            SampledGraphSAGE(graph.num_features, 256, num_classes, num_layers=3),
            graph, device=DEVICE, num_neighbors=fanouts,
            batch_size=BATCH_SIZE, inference_batch_size=INFERENCE_BATCH_SIZE,
            num_workers=NUM_WORKERS, max_epochs=MAX_EPOCHS, patience=PATIENCE,
        )
        records.append({
            'seed': seed, 'model': 'GraphSAGE', 'fanout': fanout_name,
            'best_epoch': sage_result.best_epoch,
            'validation_accuracy': sage_result.validation_accuracy,
            'test_accuracy': sage_result.test_accuracy,
            'seconds': sage_result.seconds, 'peak_cuda_mb': sage_result.peak_cuda_mb,
        })
        histories[(seed, fanout_name)] = sage_result.history

results = pd.DataFrame(records)
results.sort_values(['validation_accuracy', 'seed'], ascending=[False, True])

In [ ]:
summary = (results
    .groupby(['model', 'fanout'])[['validation_accuracy', 'test_accuracy', 'seconds', 'peak_cuda_mb']]
    .agg(['mean', 'std']))
summary

In [ ]:
plt.figure(figsize=(9, 5))
for (seed, name), history in histories.items():
    frame = pd.DataFrame(history)
    plt.plot(frame['epoch'], frame['validation_accuracy'], label=f'{name}, seed={seed}')
plt.xlabel('epoch')
plt.ylabel('validation accuracy')
plt.title('OGBN-Arxiv: feature-only vs sampled message passing')
plt.legend()
plt.show()

## 10. 怎样解释结果

不要只问哪个 accuracy 最大，还要回答：

1. GraphSAGE 是否稳定超过 MLP？这才是引用图提供增量信息的证据。
2. 更大 fanout 是否提高 validation？如果没有，额外邻居可能重复、噪声更大，或当前 fanout 已足够。
3. fanout 增大了多少运行时间与峰值 CUDA 显存？准确率增益是否值得？
4. 不同 seed 的差异来自参数初始化、batch 顺序和每轮邻居采样三种随机性。
5. 官方时间划分下的 test 是未来论文，成绩下降不等于过拟合，也可能是时间分布漂移。

本课最重要的结论不是 GraphSAGE 的具体分数，而是：**GNN 的 batch 必须同时描述监督目标和它依赖的计算子图。采样策略本身就是模型归纳偏置的一部分。**

## 11. 课后练习

1. 固定 seed，只把 `[5,5,5]` 改成 `[15,10,5]`，记录 batch 节点数、显存、速度和验证分数。
2. 把三层改成两层并同步修改 fanout，判断性能变化来自感受野还是参数量。
3. 重复读取同一个训练 loader 的第一批，检查 `n_id` 是否变化；再检查 inference loader 是否保持节点顺序。
4. 思考热门论文（高入度节点）在均匀邻居采样中是否被公平表示。后续可以由 GraphSAINT、Cluster-GCN 或重要性采样继续回答。